<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring  
**Primary metric:** Precision@50  
**Purpose:** Human decision-support for content-review prioritisation

This notebook audits the Week-5 model using an honest validation design,
a leakage review, real failure examples, and public-safe claim language.

## 1. Two paper findings + my methodology questions

### Finding 1: Growing content was longer and younger than declining content

The paper reports that content with rising impressions was, on average,
longer and younger than content with falling impressions. Growing pages
averaged approximately 3,180 words and 184 days of age, while declining
pages averaged approximately 2,311 words and 230 days of age.

### Methodology questions

**1. How was the growth or decline label created?**

The paper explains that trend direction is based on the change in
impressions between the latest 30-day period and the previous 30-day
period. I would ask whether pages with low impression counts were
filtered or stabilised before assigning the label, because small
absolute changes can create large percentage movements.

**2. Does the validation design support the claim?**

The comparison is an observational cohort comparison rather than a
predictive validation experiment. The sample supports the measured
difference within the observed portfolio, but it does not establish
that increasing word count or reducing content age will cause growth.

Client, topic, search demand, publication timing, and existing
visibility may explain part of the observed difference.

A public-safe interpretation is that longer and younger pages were
associated with stronger recent impression trends in the observed
portfolio. The result is directional and may support review
prioritisation, but it is not a causal rule.

### Finding 2: Recently refreshed mature content showed stronger measured performance

The paper reports that the 31–90 day freshness window had the strongest
stable growth-to-decline ratio. It also reports that mature content
refreshed within 30 days had higher measured health and impressions than
older content that had not been refreshed recently.

### Methodology questions

**1. Where does the refresh label come from?**

Freshness is defined as the number of days since the last content
update. I would ask what type of edit qualifies as an update. A full
rewrite, a metadata change, and an automated timestamp update may create
the same freshness value even though they represent different
interventions.

**2. Does the validation design support the claim?**

The comparison is observational. Pages selected for refresh may already
have had stronger historical visibility, greater business value, better
editorial quality, or more search demand.

This creates possible selection bias because refreshed and untouched
pages may not be directly comparable.

A stronger validation design would compare refreshed and unrefreshed
pages with similar prior impressions, position, age, topic, and client
context. A time-aware before-and-after analysis could also test whether
the measured change occurred after the refresh.

A public-safe interpretation is that recently refreshed mature pages
were associated with stronger measured performance in the observed
portfolio. This is a directional decision-support signal rather than
proof that refreshing any page will create the same result.

In [ ]:
import pandas as pd

paper_findings = pd.DataFrame(
    [
        {
            "finding": "Growing content was longer and younger",
            "reported_measure_1": "3,180 vs 2,311 average words",
            "reported_measure_2": "184 vs 230 average age in days",
            "label_source": (
                "Latest 30-day impression trend compared with "
                "the previous 30-day period"
            ),
            "evidence_type": "Observational cohort comparison",
            "safe_interpretation": (
                "Longer and younger pages were associated with "
                "stronger recent impression trends in the observed portfolio."
            ),
        },
        {
            "finding": (
                "Recently refreshed mature content showed "
                "stronger measured performance"
            ),
            "reported_measure_1": "3.2x health comparison",
            "reported_measure_2": "57x impression comparison",
            "label_source": "Days since the last recorded content update",
            "evidence_type": "Observational freshness comparison",
            "safe_interpretation": (
                "Recent refresh activity was associated with stronger "
                "measured performance among mature pages."
            ),
        },
    ]
)

display(paper_findings)

assert len(paper_findings) == 2

assert paper_findings[
    "safe_interpretation"
].str.contains(
    "associated",
    case=False,
).all()

assert paper_findings[
    "evidence_type"
].str.contains(
    "observational",
    case=False,
).all()

print(
    "Two research findings and constructive "
    "methodology questions documented."
)

## 2. My model under an honest split (before/after)

I re-run the Week-5 Logistic Regression under two validation designs.

### Before — random row split

A random row split can place pages from the same client in both training
and validation. Pages from the same client may share site structure,
measurement patterns, and editorial practices. This may make the
measured validation result optimistic.

### After — grouped client split

The grouped split places each anonymised client entirely in either
training or validation. The same client cannot occur in both sets.

This better represents the deployment question:

> Can the model rank pages belonging to a client it did not observe
> during training?

Both experiments use the same operational population, feature set,
target, Logistic Regression pipeline, test size, random seed, and
Precision@50 calculation.

The grouped result is treated as the more honest result. The random
result is retained only for the required before-and-after comparison.

In [ ]:
# Run the completed Week-5 notebook first.
# Both notebooks must be in work/notebooks/.

%run ./w05_model.ipynb

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from pathlib import Path

required_objects = [
    "model_frame",
    "feature_columns",
    "target_column",
    "group_column",
    "preprocessor",
    "SEED",
    "TEST_SIZE",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Week-5 notebook did not create: "
        + ", ".join(missing_objects)
    )

TOP_K_AUDIT = 50


def build_audit_model():
    return Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    )


def precision_at_k_audit(labels, scores, k=50):
    labels_array = np.asarray(
        labels,
        dtype=int,
    )

    scores_array = np.asarray(
        scores,
        dtype=float,
    )

    if len(labels_array) != len(scores_array):
        raise ValueError(
            "Labels and scores must have equal length."
        )

    if len(labels_array) == 0:
        raise ValueError(
            "Evaluation arrays cannot be empty."
        )

    if not np.isfinite(scores_array).all():
        raise ValueError(
            "Scores contain NaN or infinite values."
        )

    effective_k = min(
        k,
        len(labels_array),
    )

    ranked_indices = np.argsort(
        -scores_array,
        kind="mergesort",
    )[:effective_k]

    return float(
        labels_array[ranked_indices].mean()
    )


def evaluate_split(
    split_name,
    validation_frame,
    scores,
    training_rows,
    client_overlap,
):
    labels = validation_frame[
        target_column
    ].to_numpy()

    return {
        "validation_design": split_name,
        "train_rows": int(training_rows),
        "validation_rows": int(
            len(validation_frame)
        ),
        "positive_base_rate": float(
            labels.mean()
        ),
        "client_overlap": int(
            client_overlap
        ),
        "precision@50": precision_at_k_audit(
            labels,
            scores,
            TOP_K_AUDIT,
        ),
        "average_precision": float(
            average_precision_score(
                labels,
                scores,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),
    }


# ---------------------------------------------------------
# BEFORE — random row split
# ---------------------------------------------------------

all_indices = np.arange(
    len(model_frame)
)

random_train_idx, random_validation_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=model_frame[target_column],
)

random_train = model_frame.iloc[
    random_train_idx
].copy()

random_validation = model_frame.iloc[
    random_validation_idx
].copy()

random_train_clients = set(
    random_train[group_column].unique()
)

random_validation_clients = set(
    random_validation[group_column].unique()
)

random_client_overlap = len(
    random_train_clients.intersection(
        random_validation_clients
    )
)

random_model = build_audit_model()

random_model.fit(
    random_train[feature_columns],
    random_train[target_column],
)

random_scores = random_model.predict_proba(
    random_validation[feature_columns]
)[:, 1]

random_result = evaluate_split(
    split_name="Before — random row split",
    validation_frame=random_validation,
    scores=random_scores,
    training_rows=len(random_train),
    client_overlap=random_client_overlap,
)


# ---------------------------------------------------------
# AFTER — grouped client split
# ---------------------------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=SEED,
)

group_train_idx, group_validation_idx = next(
    group_splitter.split(
        model_frame[feature_columns],
        model_frame[target_column],
        groups=model_frame[group_column],
    )
)

group_train = model_frame.iloc[
    group_train_idx
].copy()

group_validation = model_frame.iloc[
    group_validation_idx
].copy()

group_train_clients = set(
    group_train[group_column].unique()
)

group_validation_clients = set(
    group_validation[group_column].unique()
)

group_client_overlap = len(
    group_train_clients.intersection(
        group_validation_clients
    )
)

assert group_client_overlap == 0

grouped_model = build_audit_model()

grouped_model.fit(
    group_train[feature_columns],
    group_train[target_column],
)

grouped_scores = grouped_model.predict_proba(
    group_validation[feature_columns]
)[:, 1]

grouped_result = evaluate_split(
    split_name="After — grouped client split",
    validation_frame=group_validation,
    scores=grouped_scores,
    training_rows=len(group_train),
    client_overlap=group_client_overlap,
)


# ---------------------------------------------------------
# Honest comparison
# ---------------------------------------------------------

split_comparison = pd.DataFrame(
    [
        random_result,
        grouped_result,
    ]
)

display(
    split_comparison.round(4)
)

random_precision_50 = float(
    random_result["precision@50"]
)

grouped_precision_50 = float(
    grouped_result["precision@50"]
)

precision_gap = (
    grouped_precision_50
    - random_precision_50
)

print(
    "Random split Precision@50:",
    round(random_precision_50, 3),
)

print(
    "Grouped split Precision@50:",
    round(grouped_precision_50, 3),
)

print(
    "Grouped minus random difference:",
    round(precision_gap, 3),
)

print(
    "Random split client overlap:",
    random_client_overlap,
)

print(
    "Grouped split client overlap:",
    group_client_overlap,
)

if precision_gap < 0:
    print(
        "Observed result: the grouped estimate is lower. "
        "This is directionally consistent with the random "
        "split being more optimistic."
    )

elif np.isclose(
    precision_gap,
    0,
):
    print(
        "Observed result: the two estimates are similar. "
        "The grouped design remains safer because client "
        "overlap is zero."
    )

else:
    print(
        "Observed result: the grouped estimate is higher "
        "on this holdout. The grouped design remains the "
        "honest design because client overlap is zero."
    )

ML09_OUTPUT_DIR = Path(
    "work/outputs/ml09"
)

ML09_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_comparison.to_csv(
    ML09_OUTPUT_DIR
    / "before_after_split_comparison.csv",
    index=False,
)

## 3. Leakage audit

The final feature set is audited against four questions:

1. Was the field available before the prediction moment?
2. Does it overlap the future outcome window?
3. Does it directly define or reproduce the target?
4. Is it an identifier or an existing decision instead of a predictive
   signal?

The feature window is March 1–15, 2026. The outcome window is
March 16–31, 2026.

Only feature-window measurements are allowed into the model.
Outcome-window measurements, identifiers, the target, and the existing
baseline score are excluded.

I also inspect real false-positive and false-negative examples from the
grouped holdout.

A false positive may waste review time or encourage an unnecessary
content change. A false negative may cause the team to miss a page that
later receives the declining proxy label.

In [ ]:
leakage_audit = pd.DataFrame(
    [
        {
            "field": "log_feature_impressions",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Derived only from feature-window impressions."
            ),
        },
        {
            "field": "feature_clicks",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Observed only in the feature window."
            ),
        },
        {
            "field": "feature_ctr",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Calculated only from feature-window "
                "clicks and impressions."
            ),
        },
        {
            "field": "feature_avg_position",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Observed only in the feature window."
            ),
        },
        {
            "field": "feature_active_days",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Counts active days only in the feature window."
            ),
        },
        {
            "field": "feature_position_volatility",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Calculated only from feature-window positions."
            ),
        },
        {
            "field": "position_band",
            "role": "model feature",
            "safe_for_model": True,
            "reason": (
                "Derived from feature-window average position."
            ),
        },
        {
            "field": "client_hash_id",
            "role": "grouping identifier",
            "safe_for_model": False,
            "reason": (
                "Used for grouped validation, not prediction."
            ),
        },
        {
            "field": "content_hash_id",
            "role": "output identifier",
            "safe_for_model": False,
            "reason": (
                "Used for deduplication and review output."
            ),
        },
        {
            "field": "outcome_impressions",
            "role": "target construction",
            "safe_for_model": False,
            "reason": (
                "Measured after the prediction moment."
            ),
        },
        {
            "field": "outcome_daily_impressions",
            "role": "target construction",
            "safe_for_model": False,
            "reason": (
                "Contains future outcome information."
            ),
        },
        {
            "field": "is_declining_proxy",
            "role": "target",
            "safe_for_model": False,
            "reason": (
                "This is the answer being predicted."
            ),
        },
        {
            "field": "baseline_review_score",
            "role": "existing decision score",
            "safe_for_model": False,
            "reason": (
                "May be compared as a baseline but "
                "must not be a model input."
            ),
        },
    ]
)

display(leakage_audit)

unsafe_fields = set(
    leakage_audit.loc[
        leakage_audit[
            "safe_for_model"
        ].eq(False),
        "field",
    ]
)

assert target_column not in feature_columns

assert "client_hash_id" not in feature_columns

assert "content_hash_id" not in feature_columns

assert "outcome_impressions" not in feature_columns

assert "outcome_daily_impressions" not in feature_columns

assert "baseline_review_score" not in feature_columns

assert set(
    feature_columns
).isdisjoint(
    unsafe_fields
)

print(
    "Leakage audit passed: "
    "no unsafe field enters the final model."
)


# ---------------------------------------------------------
# Real grouped-holdout errors
# ---------------------------------------------------------

classification_threshold = globals().get(
    "CLASSIFICATION_THRESHOLD",
    0.50,
)

grouped_predictions = (
    np.asarray(grouped_scores)
    >= classification_threshold
).astype(int)

error_columns = [
    column
    for column in [
        "feature_impressions",
        "feature_clicks",
        "feature_ctr",
        "feature_avg_position",
        "feature_active_days",
        "feature_position_volatility",
        "position_band",
        target_column,
    ]
    if column in group_validation.columns
]

error_frame = group_validation[
    error_columns
].copy()

error_frame[
    "model_score"
] = grouped_scores

error_frame[
    "prediction"
] = grouped_predictions

error_frame[
    "error_type"
] = np.select(
    [
        (
            error_frame[target_column].eq(0)
            & error_frame["prediction"].eq(1)
        ),
        (
            error_frame[target_column].eq(1)
            & error_frame["prediction"].eq(0)
        ),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct",
)

false_positives = (
    error_frame[
        error_frame[
            "error_type"
        ].eq("false_positive")
    ]
    .sort_values(
        "model_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

false_negatives = (
    error_frame[
        error_frame[
            "error_type"
        ].eq("false_negative")
    ]
    .assign(
        distance_from_threshold=lambda frame: (
            frame["model_score"]
            - classification_threshold
        ).abs()
    )
    .sort_values(
        "distance_from_threshold"
    )
    .reset_index(drop=True)
)

error_counts = (
    error_frame[
        "error_type"
    ]
    .value_counts()
    .rename_axis(
        "error_type"
    )
    .reset_index(
        name="rows"
    )
)

display(error_counts)

print(
    "Highest-scoring false positives:"
)

display(
    false_positives.head(10)
)

print(
    "False negatives closest to the threshold:"
)

display(
    false_negatives.head(10)
)

assert len(false_positives) > 0
assert len(false_negatives) > 0

leakage_audit.to_csv(
    ML09_OUTPUT_DIR
    / "leakage_audit.csv",
    index=False,
)

false_positives.to_csv(
    ML09_OUTPUT_DIR
    / "false_positives_grouped.csv",
    index=False,
)

false_negatives.to_csv(
    ML09_OUTPUT_DIR
    / "false_negatives_grouped.csv",
    index=False,
)

## 4. Claim rewrite

### My boldest original claim

> Logistic Regression predicts which pages need a refresh.

### Public-safe rewrite

On the grouped client holdout used in this notebook, Logistic Regression
measured a specific Precision@50 on unseen client groups. This observed
result is directional and is intended for decision-support.

It supports prioritising pages for human review, but it does not prove
that every selected page requires a refresh or that changing an
individual feature will improve future search performance.

### Feature claim rewrite

**Unsafe claim:** CTR causes page decline.

**Safe claim:** The fitted model may rely strongly on feature-window CTR
when ranking pages on the grouped holdout. This describes measured model
behaviour under this validation design and does not establish causation.

### Deployment claim rewrite

**Unsafe claim:** The model should automatically decide which pages to
refresh.

**Safe claim:** The model is a decision-support ranking tool. A human
reviewer should inspect the ranked pages and decide whether the
appropriate action is refresh, expansion, CTR review, protection,
monitoring, or no immediate action.

In [ ]:
claim_audit = pd.DataFrame(
    [
        {
            "claim_type": "model performance",
            "unsafe_claim": (
                "The model predicts which pages need a refresh."
            ),
            "safe_claim": (
                f"On the grouped client holdout, Logistic "
                f"Regression measured Precision@50="
                f"{grouped_precision_50:.3f}. The result is "
                "observed, measured, directional, and intended "
                "for decision-support."
            ),
        },
        {
            "claim_type": "feature interpretation",
            "unsafe_claim": (
                "CTR causes page decline."
            ),
            "safe_claim": (
                "Feature importance describes measured model "
                "reliance under this split and does not "
                "establish causation."
            ),
        },
        {
            "claim_type": "deployment",
            "unsafe_claim": (
                "The model should automate refresh decisions."
            ),
            "safe_claim": (
                "The model supports a human review queue; "
                "human review remains required."
            ),
        },
    ]
)

display(claim_audit)

safe_language = " ".join(
    claim_audit[
        "safe_claim"
    ].str.lower()
)

required_safe_words = [
    "observed",
    "measured",
    "directional",
    "decision-support",
    "human review",
]

for word in required_safe_words:
    assert word in safe_language, (
        f"Missing safe language: {word}"
    )

claim_audit.to_csv(
    ML09_OUTPUT_DIR
    / "claim_rewrite_audit.csv",
    index=False,
)

print(
    "Claim rewrite audit completed."
)

## Self-check

Before submission, I confirm that:

- every required section contains markdown reasoning and supporting code;
- random-row and grouped-client validation are compared;
- the grouped validation split has zero client overlap;
- outcome-window fields, identifiers, the target, and the existing
  baseline score are excluded from the final features;
- real false-positive and false-negative examples are inspected;
- no client names, domains, URLs, or private queries are displayed;
- claims use observed, measured, directional, and decision-support
  language;
- outputs and the grouped model are saved under `work/outputs/ml09/`.

In [ ]:
import joblib

MODEL_DIR_ML09 = (
    ML09_OUTPUT_DIR
    / "models"
)

MODEL_DIR_ML09.mkdir(
    parents=True,
    exist_ok=True,
)

model_path = (
    MODEL_DIR_ML09
    / "ml09_grouped_logistic_regression.joblib"
)

joblib.dump(
    grouped_model,
    model_path,
)

required_files = [
    ML09_OUTPUT_DIR
    / "before_after_split_comparison.csv",

    ML09_OUTPUT_DIR
    / "leakage_audit.csv",

    ML09_OUTPUT_DIR
    / "false_positives_grouped.csv",

    ML09_OUTPUT_DIR
    / "false_negatives_grouped.csv",

    ML09_OUTPUT_DIR
    / "claim_rewrite_audit.csv",

    model_path,
]

self_checks = {
    "Two paper findings documented": (
        len(paper_findings) == 2
    ),

    "Before and after validation completed": (
        len(split_comparison) == 2
    ),

    "Grouped client overlap equals zero": (
        group_client_overlap == 0
    ),

    "Random base rate printed": (
        0
        <= random_result["positive_base_rate"]
        <= 1
    ),

    "Grouped base rate printed": (
        0
        <= grouped_result["positive_base_rate"]
        <= 1
    ),

    "Leakage audit completed": (
        len(leakage_audit) >= 10
    ),

    "Target excluded": (
        target_column
        not in feature_columns
    ),

    "Identifiers excluded": (
        "client_hash_id"
        not in feature_columns
        and
        "content_hash_id"
        not in feature_columns
    ),

    "Outcome fields excluded": (
        "outcome_impressions"
        not in feature_columns
        and
        "outcome_daily_impressions"
        not in feature_columns
    ),

    "Existing decision score excluded": (
        "baseline_review_score"
        not in feature_columns
    ),

    "False positives inspected": (
        len(false_positives) > 0
    ),

    "False negatives inspected": (
        len(false_negatives) > 0
    ),

    "Grouped Precision@50 valid": (
        0
        <= grouped_precision_50
        <= 1
    ),

    "Safe language present": all(
        word in safe_language
        for word in required_safe_words
    ),

    "Required outputs saved": all(
        path.exists()
        for path in required_files
    ),
}

self_check_df = pd.DataFrame(
    {
        "check": list(
            self_checks.keys()
        ),
        "passed": list(
            self_checks.values()
        ),
    }
)

display(self_check_df)

failed_checks = [
    check_name
    for check_name, passed
    in self_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Self-check failed: "
        + ", ".join(failed_checks)
    )

print(
    "Saved grouped model:",
    model_path,
)

print(
    "Saved audit outputs:",
    ML09_OUTPUT_DIR,
)

print(
    "All ML-09 self-checks passed."
)